# FETAL-CHD: 44-model Kaggle benchmark

This notebook loads the CARDIUM dataset and a **real 44-row model manifest**, downloads and caches each checkpoint, evaluates all available models on the same test split, and packages the results and weights into a ZIP file.

> The manifest must contain the exact checkpoint URLs and licenses. Do not replace unavailable checkpoints with invented URLs. Failed/private models are recorded as failed and are not counted as valid scores.

In [ ]:
# Kaggle Settings: enable Internet and GPU if the selected models require them.
# The code below discovers the real mounted paths, including Kaggle's
# generated dataset names and extra nested folders.

from pathlib import Path
import torch

input_root = Path("/kaggle/input")
manifest_candidates = sorted(input_root.rglob("models.csv"))
if len(manifest_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one models.csv under /kaggle/input; found: "
        + ", ".join(str(p) for p in manifest_candidates)
    )
MANIFEST = str(manifest_candidates[0])

test_candidates = sorted(input_root.rglob("fold_1/test/CHD"))
if len(test_candidates) != 1:
    raise FileNotFoundError(
+        "Expected one CARDIUM fold_1/test/CHD directory; found: "
+        + ", ".join(str(p) for p in test_candidates)
+    )
+    DATA_ROOT = str(test_candidates[0].parents[3])
EXPECTED_MODELS = 44
FOLD = 1
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR = "/kaggle/working/model_cache"
RESULTS_CSV = "/kaggle/working/benchmark_results.csv"
BUNDLE_ZIP = "/kaggle/working/fetal_chd_models_and_results.zip"
print({"device": DEVICE, "data": DATA_ROOT, "manifest": MANIFEST})

In [ ]:
!pip install -q timm scikit-learn tqdm pillow torchvision

In [ ]:
# Kaggle cannot clone private repositories or Copilot-generated URLs.
# Preferred: attach a Kaggle Dataset named fetal-chd-code containing
# kaggle_benchmark.py and data.py. The fallback downloads public raw files.

import os, sys, urllib.request
REPO_DIR = "/kaggle/working/FETAL-CHD"
os.makedirs(REPO_DIR, exist_ok=True)
attached_code = "/kaggle/input/fetal-chd-code"
for filename in ("kaggle_benchmark.py", "data.py"):
    destination = os.path.join(REPO_DIR, filename)
    attached = os.path.join(attached_code, filename)
    if os.path.exists(attached):
        import shutil
        shutil.copy2(attached, destination)
    elif not os.path.exists(destination):
        url = "https://raw.githubusercontent.com/Gokzz-glitch/FETAL-CHD/main/" + filename
        try:
            urllib.request.urlretrieve(url, destination)
        except Exception as error:
            raise RuntimeError(
                "Attach a Kaggle Dataset named fetal-chd-code containing "
                "kaggle_benchmark.py and data.py, then rerun this cell."
            ) from error
sys.path.insert(0, REPO_DIR)
print(REPO_DIR)

In [ ]:
from pathlib import Path
import csv

manifest_path = Path(MANIFEST)
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}")
with manifest_path.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
required = {"name", "loader", "architecture", "checkpoint_url", "filename"}
if len(rows) != EXPECTED_MODELS:
    raise ValueError(f"Expected exactly {EXPECTED_MODELS} models, found {len(rows)}")
if required - set(rows[0]):
    raise ValueError(f"Manifest missing columns: {sorted(required - set(rows[0]))}")
print(f"Validated {len(rows)} manifest rows")

In [ ]:
# Verify the CARDIUM test split before downloading large weights.
from data import CARDIUMDataset
dataset = CARDIUMDataset(DATA_ROOT, split="test", fold=FOLD)
if len(dataset) == 0:
    raise ValueError(
        "No test images found. Expected "
        "cardium_images/cardium_images/fold_N/test/{CHD,Non_CHD}."
    )
print(f"CARDIUM test images: {len(dataset)}")

In [ ]:
# Download/cache weights and evaluate all 44 models.
import sys, subprocess
cmd = [
    sys.executable, os.path.join(REPO_DIR, "kaggle_benchmark.py"),
    "--data-root", DATA_ROOT,
    "--manifest", MANIFEST,
    "--expected-models", str(EXPECTED_MODELS),
    "--fold", str(FOLD),
    "--batch-size", str(BATCH_SIZE),
    "--device", DEVICE,
    "--cache", CACHE_DIR,
    "--out", RESULTS_CSV,
]
subprocess.run(cmd, check=True)

In [ ]:
# Inspect and rank successful models.
import pandas as pd
results = pd.read_csv(RESULTS_CSV)
display(results.sort_values("roc_auc", ascending=False, na_position="last"))
print(results["status"].value_counts(dropna=False))

In [ ]:
# Package results and downloaded model weights for download from Kaggle.
import shutil
bundle_dir = Path("/kaggle/working/fetal_chd_bundle")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True)
shutil.copy2(RESULTS_CSV, bundle_dir / "benchmark_results.csv")
shutil.copy2(MANIFEST, bundle_dir / "models.csv")
shutil.copytree(CACHE_DIR, bundle_dir / "weights")
shutil.make_archive(BUNDLE_ZIP[:-4], "zip", bundle_dir)
print(f"Download this artifact: {BUNDLE_ZIP}")

## Saving to Google Drive

Kaggle cannot directly mount Google Drive. Download `/kaggle/working/fetal_chd_models_and_results.zip` from the notebook Output panel, then upload it to Drive. For a persistent Kaggle copy, save the notebook output as a Kaggle Dataset.